In [0]:
# Catalog Name
catalog = 'flight_project'

# Source Schema
source_schema = 'silver'

# Source Object
source_object = 'silver_bookings'

# CDC Column
cdc_column = 'modified_date'

# Backdated Refresh
backdated_refresh = ''

# Source Fact Table
fact_table = f'{catalog}.{source_schema}.{source_object}'

# Target Schema
target_schema = 'gold'

# Target Object
target_object = 'fact_bookings'

# Fact Key Cols List
fact_key_cols = ['DimPassengersKey', 'DimFlightsKey', 'DimAirportsKey', 'booking_date']

In [0]:
dimensions = [
    {
        'table' : f'{catalog}.{target_schema}.dim_passengers',
        'alias' : 'Passengers',
        'join_keys' : [('passenger_id', 'passenger_id')] #(fact_col, dim_col)
    },
    {
        'table' : f'{catalog}.{target_schema}.dim_flights',
        'alias' : 'Flights',
        'join_keys' : [('flight_id', 'flight_id')] #(fact_col, dim_col)
    },
    {
        'table' : f'{catalog}.{target_schema}.dim_airports',
        'alias' : 'Airports',
        'join_keys' : [('airport_id', 'airport_id')] #(fact_col, dim_col)
    },
]

# Columns you want to keep from Fact Table (besides the surrogate keys)
fact_columns = ['amount', 'booking_date', 'modified_date']

### **Last Load Date**

In [0]:
# No Backdated Refresh
if len(backdated_refresh) == 0:

  # if Table Exists in the Destination
  if spark.catalog.tableExists(f'{catalog}.{target_schema}.{target_object}'):

    last_load = spark.sql(f'SELECT max({cdc_column}) FROM {catalog}.{target_schema}.{target_object}').collect()[0][0]

  else:

    last_load = '1900-01-01 00:00:00'
# Yes Backdated Refresh
else:
  last_load = backdated_refresh

# Test the last load
last_load

### **Dynamic Fact Query [Bring Keys]**

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, processing_date):
    fact_alias = "f"

    # Base Columns to Select
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    # Build Joins Dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim['table']
        alias = dim['alias']
        table_name = table_full.split('.')[-1]
        surrogate_key = f'{alias}.Dim{alias}Key'
        select_cols.append(surrogate_key)

        #Build on Clause
        on_condition = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for fk, dk in dim["join_keys"]
        ]
        join_clause = f'LEFT JOIN {table_full} {alias} ON ' + " AND ".join(on_condition)
        join_clauses.append(join_clause)

    # Final SELECT and JOIN clauses
    select_clause = ",\n    ".join(select_cols)
    joins = "\n".join(join_clauses)

    # WHERE clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_column} >= DATE('{last_load}')"

    # Final Query
    query = f"""
SELECT
    {select_clause}
FROM {fact_table} {fact_alias}
{joins}
WHERE {where_clause}
""".strip()

    return query


In [0]:
query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, last_load)

In [0]:
print(query)

### **DF_FACT**

In [0]:
df_fact = spark.sql(query)
df_fact.display()

In [0]:
display(df_fact.groupBy('DimPassengersKey', 'DimFlightsKey', 'DimAirportsKey').count().filter('count > 1'))

### **UPSERT**

In [0]:
# Fact Key Columns Merge Condition
fact_key_cols_str = " AND ".join([f"src.{col} = trg.{col}" for col in fact_key_cols])
fact_key_cols_str

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f'{catalog}.{target_schema}.{target_object}'):
    
    dlt_obj = DeltaTable.forName(spark, f'{catalog}.{target_schema}.{target_object}')
    
    dlt_obj.alias('trg').merge(df_fact.alias('src'), fact_key_cols_str)\
                        .whenMatchedUpdateAll(condition = f'src.{cdc_column} >= trg.{cdc_column}')\
                        .whenNotMatchedInsertAll()\
                        .execute()

else:
    
    df_fact.write.format('delta')\
                  .mode('append')\
                  .saveAsTable(f'{catalog}.{target_schema}.{target_object}')


In [0]:
%sql
SELECT *
FROM flight_project.gold.fact_bookings

### **To check if there's duplicate**

In [0]:
df = spark.sql('SELECT * FROM flight_project.gold.dim_passengers').groupBy('DimPassengersKey').count().filter('count > 1')
display(df)